In [ ]:
# Lesson 4: Practical Deep Learning for Coders 2022
# 微调预训练模型（NLP：把两段文本的相似度当回归问题，用 HuggingFace Transformers）


In [ ]:
# ⚠ 数据源说明：本课实际用的是 Kaggle "US Patent Phrase to Phrase Matching" 数据集，
#   train.csv 里含 anchor / target / context / score 列。下面的 MNIST_SAMPLE 只是占位，
#   真正运行前请换成该 patent 数据集所在路径，否则 read_csv('train.csv') 会找不到文件。
# （原来还误 import 了 tokenize.tokenize 和 sympy.abc.alpha，都没用到，已删除）
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fastai.data.external import *

path = untar_data(URLs.MNIST_SAMPLE)
df = pd.read_csv(path/'train.csv')


In [ ]:
# 看非数值（object）列的统计信息
df.describe(include='object')


In [ ]:
# 把三段文本拼成一个 input 字段，一起喂给语言模型
df['input'] = 'TEXT1:' + df.context + ';TEXT2:' + df.target + ";ANC1:" + df.anchor


In [ ]:
# 看看拼好的 input
df.input.head()


In [ ]:
# 转成 HuggingFace 的 Dataset 格式
from datasets import *
ds = Dataset.from_pandas(df)
ds


In [ ]:
# 选用的预训练模型：微软 deberta-v3-small
model_nm = 'microsoft/deberta-v3-small'


In [ ]:
# 加载与该模型配套的分词器
from transformers import AutoModelForSequenceClassification, AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_nm)


In [ ]:
# 试试分词效果
tokz.tokenize('G day folks, im jeremy from fast.ai')


In [ ]:
# 再试一句带生僻词的
tokz.tokenize('a platypus is an ornithorhynchus anatinus.')


In [ ]:
# 定义分词函数：对每行的 input 字段分词
def tok_func(x): return tokz(x['input'])


In [ ]:
# 批量分词整个数据集
tok_ds = ds.map(tok_func, batched=True)


In [ ]:
# 看第一行：原文 input 和它对应的 token id
row = tok_ds[0]
row['input'], row['input_ids']


In [ ]:
# 查某个 token 的 id；注意 deberta 用 '▁'(U+2581) 表示词首空格，不是普通下划线
tokz.vocab['▁of']   # 修正：原来是 '_of'（普通下划线），会 KeyError


In [ ]:
# Trainer 要求标签列名为 'labels'，把 'score' 改名
tok_ds = tok_ds.rename_columns({'score': 'labels'})   # 修正：按字典改名要用 rename_columns（复数）


In [ ]:
# 把 tokenize 后的数据集切成 训练 / 测试 两份（下面 Trainer 需要 dds['train'] 和 dds['test']）
dds = tok_ds.train_test_split(0.25, seed=42)
dds


In [ ]:
# 读入验证 / 测试集
eval_df = pd.read_csv(path/'test.csv')   # 修正：原来是 'test.cvs' 拼写错误
eval_df.describe()


In [ ]:
# 通用绘图工具（原课程有定义，这里补上，供上面的 plot_poly 使用）
import torch
def plot_function(f, title=None, min=-2.1, max=2.1, color='r', ylim=None):
    x = torch.linspace(min, max, 100)[:, None]
    if ylim: plt.ylim(ylim)
    plt.plot(x, f(x), color)
    if title is not None: plt.title(title)


In [ ]:
# 多项式回归拟合示例（degree 越高越容易过拟合）；注意：调用前需先有全局的 x, y
from sklearn.linear_model import *
from sklearn.preprocessing import *
from sklearn.pipeline import *
def plot_poly(degree):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(x, y)
    plt.scatter(x, y)
    plot_function(model.predict)


In [ ]:
# —— 相关系数小节：加州房价数据 ——
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
housing = housing['data'].join(housing['target']).sample(1000, random_state=52)
housing.head()


In [ ]:
# 各列两两之间的相关系数矩阵
np.set_printoptions(precision=2, suppress=True)
np.corrcoef(housing, rowvar=False)


In [ ]:
# 收入 与 房价 的相关系数矩阵
np.corrcoef(housing.MedInc, housing.MedHouseVal)


In [ ]:
# 取相关系数矩阵里 [0][1] 的那个值（两变量之间的相关系数标量）
def corr(x, y): return np.corrcoef(x, y)[0][1]   # 修正：原来只取 [0]，返回的是数组，后面 f-string :.2f 会报错
corr(housing.MedInc, housing.MedHouseVal)


In [ ]:
# 画散点图并在标题里标注相关系数
def show_corr(df, a, b):
    x, y = df[a], df[b]
    plt.scatter(x, y, alpha=0.5, s=4)   # 修正：原来是 plt.subplots，那不是画散点的函数
    plt.title(f'{a} vs {b}; r:{corr(x, y):.2f}')


In [ ]:
# 收入 vs 房价
show_corr(housing, 'MedInc', 'MedHouseVal')


In [ ]:
# 收入 vs 平均房间数
show_corr(housing, 'MedInc', 'AveRooms')


In [ ]:
# 去掉平均房间数 >15 的异常值后再看（用过滤后的 subset）
subset = housing[housing.AveRooms < 15]
show_corr(subset, "MedInc", "AveRooms")   # 修正：原来传的是 housing，没用上过滤后的 subset


In [ ]:
# 训练时用的评估指标：皮尔逊相关系数
def corr_d(eval_pred): return {"pearson": corr(*eval_pred)}


In [ ]:
# HuggingFace 训练相关的类
from transformers import TrainingArguments, Trainer


In [ ]:
# 超参数：批大小 / 轮数 / 学习率
bs = 128
epochs = 4
lr = 8e-5


In [ ]:
# 训练参数配置
args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, lr_scheduler_type='cosine',
                         fp16=True, eval_strategy='epoch', per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
                         num_train_epochs=epochs, weight_decay=0.01, report_to='none')


In [ ]:
# 建模型（回归任务，num_labels=1）并组装 Trainer
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=1)
trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds["test"],
                  tokenizer=tokz, compute_metrics=corr_d)


In [ ]:
# 开始训练
trainer.train()


In [ ]:
# 预测前，验证集也要和训练集一样处理：拼 input 列 -> 转 Dataset -> 分词
eval_df['input'] = 'TEXT1:' + eval_df.context + ';TEXT2:' + eval_df.target + ";ANC1:" + eval_df.anchor
eval_ds = Dataset.from_pandas(eval_df).map(tok_func, batched=True)
preds = trainer.predict(eval_ds).predictions.astype(float)   # 修正：原来把 pandas 的 eval_df 直接传进去，应传分词后的 Dataset
preds
